# Справочное обогащение

Тематический этап подготовки Европы 2019–2023. Выполняйте ячейки сверху вниз в окружении проекта. Каждый ноутбук соответствует одному шагу; диагностика и применение согласованного изменения разделены. Входы, выходы и порядок описаны в [карте подготовки](README.md). Сохраненные результаты перенесены из выполненного общего ноутбука; сам перенос данных не преобразует.

## Шаг 5 – Согласованное справочное обогащение

Согласовано добавление производного описания для 18 прямо сопоставленных значений. Исходные `icaoaircrafttype`, `typecode` и все остальные существующие поля сохраняются. Новая копия – `europe/data/working/02_aircraft_reference_enriched.parquet`.

Добавляем `aircraft_description_reference`, `aircraft_reference_status`, `aircraft_reference_source` и `aircraft_reference_checked_date`. Для 5 888 сопоставленных записей источник – публичная таблица [ICAO Doc 8643](https://www.icao.int/operational-safety/doc-8643-aircraft-type-designators/search), снимок 08.10.2026. Производное описание для остальных записей остается null: статус различает неразрешенные значения, отсутствующее исходное описание и значения вне текущего разбора.

Историческое соответствие справочника за 2019–2023 годы не подтверждено. Производное поле не означает независимое подтверждение типа конкретного борта. Сначала проверяем SHA256 входной копии и справочного снимка; после записи сравниваем все существующие поля с входом и проверяем производные значения.

In [1]:
# Запускаем согласованное справочное обогащение отдельным скриптом
from pathlib import Path
import runpy
import json
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# run_path загружает функции скрипта; main выполняет преобразование и проверки
module = runpy.run_path(str(ROOT / 'europe/scripts/apply_aircraft_reference.py'))
changes = module['main']()
display(changes)
# Показываем журнал: сохранность строк, меток и всех существующих полей
journal = json.loads((ROOT / 'europe/reports/preparation/04_aircraft_reference_applied.json').read_text(encoding='utf8'))
print('Записей до и после:', journal['rows_before'], journal['rows_after'])
print('Отмеченных уходов до и после:', journal['go_arounds_before'], journal['go_arounds_after'])
print('Справочное описание добавлено:', journal['rows_enriched'])
print('Статусы:', journal['status_counts'])
print('Все существующие поля неизменны:', journal['existing_columns_unchanged'])
print('Официальный источник:', journal['reference_url'])
print('Дата проверки:', journal['reference_checked_date'])

Rows preserved: 7984254 Rows enriched: 5888


,source_icaoaircrafttype,reference_description,rows_enriched,go_arounds
0,A21N,L2J,705,7
1,A319,L2J,5,0
2,A332,L2J,1140,1
3,AT76,L2T,25,0
4,B734,L2J,481,0
5,BE4W,L2J,68,0
6,C68A,L2J,357,0
7,E300,L1P,28,0
8,F2TH,L2J,137,0
9,FA7X,L3J,126,0


Записей до и после: 7984254 7984254
Отмеченных уходов до и после: 26253 26253
Справочное описание добавлено: 5888
Статусы: {'not_reviewed': 6575611, 'source_description_missing': 1402236, 'direct_unique_reference_match': 5888, 'unresolved': 519}
Все существующие поля неизменны: True
Официальный источник: https://doc8643.icao.int/External/AircraftTypes
Дата проверки: 2026-10-08


## Следующие действия требуют отдельного согласования

Выполнены только нормализация точных пустых строк в трех полях и согласованное справочное обогащение 18 значений. Исходные коды не перезаписаны. Пять неразрешенных значений не исправлены, записи не исключены. Следующий разбор и любое новое правило согласуются отдельно.